In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:28:37,601]::2022985491::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 50

# Multi-seed setting
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# TimeSformer Layer-wise Linear Probe | Multi-seed
# ============================================================
from vit_probe_epoch50_patience10 import TimeSformer, TimeSformerLinearProbe

backbone_checkpoint_path = './save_timesformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_timesformer_folder/timesformer_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
criterion = torch.nn.MSELoss().to(device)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mse = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):
    for seed in SEEDS:
        print('\n' + '=' * 80)
        print(f'Start TimeSformer probe | seed = {seed}')
        print('=' * 80)
        set_seed(seed)
        train_generator = torch.Generator(); train_generator.manual_seed(seed)
        val_generator = torch.Generator(); val_generator.manual_seed(seed + 10000)
        train_dataloader = DataLoader(dataset=train_set_data,batch_size=batchsz,shuffle=True,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=train_generator)
        val_dataloader = DataLoader(dataset=val_set_data,batch_size=batchsz,shuffle=False,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=val_generator)

        backbone = load_backbone_checkpoint(TimeSformer(), backbone_checkpoint_path)
        model = TimeSformerLinearProbe(backbone=backbone, num_outputs=5).to(device)
        optimizer = torch.optim.AdamW(model.probe_heads.parameters(), lr=lr)
        probe_names = list(model.probe_heads.keys())
        seed_best = {name:{'mse':float('inf'),'epoch':0} for name in probe_names}
        seed_best_mean_mse = float('inf'); no_improve_count = 0
        print('Backbone trainable params:', sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
        print('Probe dimensions:', model.probe_dims)

        for epoch in range(epochs):
            model.train()
            train_sq_sum = {name:0.0 for name in probe_names}; train_count = {name:0 for name in probe_names}
            for audio, fullshot, big_five_data in tqdm(train_dataloader, desc=f'Probe Seed {seed} Train {epoch+1}/{epochs}'):
                fullshot = fullshot.permute(0,4,1,2,3)
                big_five_data = big_five_data.permute(0,2,1).squeeze()
                audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                optimizer.zero_grad(); outputs = model(fullshot,audio)
                losses = {name:criterion(outputs[name],big_five_data) for name in probe_names}
                torch.stack([losses[name] for name in probe_names]).mean().backward(); optimizer.step()
                with torch.no_grad():
                    for name in probe_names:
                        train_sq_sum[name] += torch.square(outputs[name]-big_five_data).sum().item(); train_count[name] += big_five_data.numel()
            train_mse = {name:train_sq_sum[name]/train_count[name] for name in probe_names}

            model.eval(); val_sq_sum = {name:0.0 for name in probe_names}; val_count = {name:0 for name in probe_names}
            val_trait_sq_sum = {name:np.zeros(5,dtype=np.float64) for name in probe_names}; val_sample_count = 0
            with torch.no_grad():
                for audio, fullshot, big_five_data in tqdm(val_dataloader, desc=f'Probe Seed {seed} Val {epoch+1}/{epochs}'):
                    fullshot = fullshot.permute(0,4,1,2,3)
                    big_five_data = big_five_data.permute(0,2,1).squeeze()
                    audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                    outputs = model(fullshot,audio)
                    if big_five_data.ndim == 1: big_five_data = big_five_data.unsqueeze(0)
                    val_sample_count += big_five_data.shape[0]
                    for name in probe_names:
                        sq_err = torch.square(outputs[name]-big_five_data)
                        val_sq_sum[name] += sq_err.sum().item(); val_count[name] += big_five_data.numel()
                        val_trait_sq_sum[name] += sq_err.sum(dim=0).detach().cpu().numpy()
            val_mse = {name:val_sq_sum[name]/val_count[name] for name in probe_names}
            val_trait_mse = {name:val_trait_sq_sum[name]/val_sample_count for name in probe_names}
            mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))
            for name in probe_names:
                if val_mse[name] < seed_best[name]['mse']:
                    seed_best[name]['mse'] = float(val_mse[name]); seed_best[name]['epoch'] = epoch + 1
            if mean_val_mse < seed_best_mean_mse:
                seed_best_mean_mse = mean_val_mse; no_improve_count = 0
                if mean_val_mse < global_best_mean_mse:
                    global_best_mean_mse = mean_val_mse; global_best_seed = seed; global_best_epoch = epoch + 1
                    torch.save({'seed':seed,'epoch':epoch+1,'probe_model':model.probe_heads.state_dict(),'mean_val_mse':mean_val_mse,'per_probe_val_mse':val_mse,'probe_dims':model.probe_dims},probe_best_model_path)
            else: no_improve_count += 1

            print(f'\\nSeed {seed} | Epoch {epoch+1:03d} | Mean Val MSE={mean_val_mse:.6f} | Best Mean Val MSE={seed_best_mean_mse:.6f} | Patience={no_improve_count}/{patience}')
            print('{:<18} {:>14} {:>14} {:>14}'.format('Representation','Train MSE','Val MSE','Best MSE'))
            print('-'*64)
            for name in probe_names:
                print('{:<18} {:>14.6f} {:>14.6f} {:>14.6f}'.format(name,train_mse[name],val_mse[name],seed_best[name]['mse']))
                o_mse,c_mse,e_mse,a_mse,n_mse = val_trait_mse[name]
                print('  O MSE={:.6f} | C MSE={:.6f} | E MSE={:.6f} | A MSE={:.6f} | N MSE={:.6f}'.format(o_mse,c_mse,e_mse,a_mse,n_mse))
            if no_improve_count >= patience:
                logger.info('Probe early stopping | seed={} | epoch={} | best mean MSE={:.6f}'.format(seed,epoch+1,seed_best_mean_mse)); break
            torch.cuda.empty_cache()

        seed_results.append({'seed':seed,'representations':{name:{'best_epoch':seed_best[name]['epoch'],'best_mse':seed_best[name]['mse'],'best_1_mae':1.0-seed_best[name]['mse']} for name in probe_names}})
        del model, backbone, optimizer, train_dataloader, val_dataloader
        gc.collect(); torch.cuda.empty_cache()

print('\n' + '='*72)
print('FINAL TIMESFORMER FCCC LAYERWISE LINEAR PROBE RESULTS -- UDIVA / MSE')
print('='*72)
print('{:<18} {:>16} {:>16}'.format('Representation','MSE Mean','MSE Std'))
print('-'*72)
for name in probe_names:
    mses = np.array(
        [r['representations'][name]['best_mse'] for r in seed_results],
        dtype=np.float64
    )
    print('{:<18} {:>16.6f} {:>16.6f}'.format(
        name, mses.mean(), mses.std(ddof=1)
    ))
print('-'*72)
print('Global best mean-probe checkpoint: seed={}, epoch={}, mean MSE={:.6f}'.format(
    global_best_seed, global_best_epoch, global_best_mean_mse
))
print('Saved probe checkpoint:', probe_best_model_path)
print('='*72)



Start TimeSformer probe | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable params: 0
Probe dimensions: {'block1': 768, 'block2': 768, 'block3': 768, 'block4': 768, 'block5': 768, 'block6': 768, 'block7': 768, 'block8': 768, 'block9': 768, 'block10': 768, 'block11': 768, 'block12': 768, 'final_video': 768, 'final_cross': 768, 'final_fused': 768}


Probe Seed 42 Val 1/50: 100%|██████████| 191/191 [00:19<00:00,  9.93it/s]


\nSeed 42 | Epoch 001 | Mean Val MSE=1.363757 | Best Mean Val MSE=1.363757 | Patience=0/10
Representation          Train MSE        Val MSE       Best MSE
----------------------------------------------------------------
block1                   0.218962       1.617181       1.617181
  O MSE=1.627284 | C MSE=1.360603 | E MSE=1.557006 | A MSE=1.914833 | N MSE=1.626176
block2                   0.251042       1.502168       1.502168
  O MSE=1.478810 | C MSE=1.388224 | E MSE=1.206646 | A MSE=1.972328 | N MSE=1.464831
block3                   0.194002       1.458738       1.458738
  O MSE=1.396459 | C MSE=1.324125 | E MSE=1.210565 | A MSE=1.890640 | N MSE=1.471903
block4                   0.261788       1.423461       1.423461
  O MSE=1.313290 | C MSE=1.293064 | E MSE=1.197205 | A MSE=1.883883 | N MSE=1.429863
block5                   0.267853       1.395616       1.395616
  O MSE=1.264933 | C MSE=1.335180 | E MSE=1.189458 | A MSE=1.831940 | N MSE=1.356568
block6                   0.194898  

Probe Seed 42 Val 2/50: 100%|██████████| 191/191 [00:19<00:00,  9.94it/s]


\nSeed 42 | Epoch 002 | Mean Val MSE=1.386618 | Best Mean Val MSE=1.363757 | Patience=1/10
Representation          Train MSE        Val MSE       Best MSE
----------------------------------------------------------------
block1                   0.106760       1.556035       1.556035
  O MSE=1.560207 | C MSE=1.321524 | E MSE=1.426872 | A MSE=1.851336 | N MSE=1.620238
block2                   0.071968       1.467363       1.467363
  O MSE=1.434694 | C MSE=1.386424 | E MSE=1.135397 | A MSE=1.910561 | N MSE=1.469737
block3                   0.045282       1.440848       1.440848
  O MSE=1.358629 | C MSE=1.360535 | E MSE=1.162346 | A MSE=1.865309 | N MSE=1.457423
block4                   0.037285       1.411152       1.411152
  O MSE=1.307752 | C MSE=1.318457 | E MSE=1.156284 | A MSE=1.843326 | N MSE=1.429943
block5                   0.028544       1.400322       1.395616
  O MSE=1.270198 | C MSE=1.378579 | E MSE=1.145293 | A MSE=1.824811 | N MSE=1.382730
block6                   0.026811  

...

Probe Seed 42 Val 10/50: 100%|██████████| 191/191 [00:20<00:00,  9.43it/s]


\nSeed 42 | Epoch 010 | Mean Val MSE=1.375349 | Best Mean Val MSE=1.363757 | Patience=9/10
Representation          Train MSE        Val MSE       Best MSE
----------------------------------------------------------------
block1                   0.049491       1.451966       1.442900
  O MSE=1.355349 | C MSE=1.298505 | E MSE=1.301379 | A MSE=1.743501 | N MSE=1.561098
block2                   0.033913       1.416052       1.400695
  O MSE=1.323619 | C MSE=1.416328 | E MSE=1.059898 | A MSE=1.822134 | N MSE=1.458281
block3                   0.025516       1.398558       1.391495
  O MSE=1.290577 | C MSE=1.386687 | E MSE=1.070318 | A MSE=1.834156 | N MSE=1.411050
block4                   0.022910       1.381760       1.371801
  O MSE=1.252437 | C MSE=1.376161 | E MSE=1.077104 | A MSE=1.793843 | N MSE=1.409254
block5                   0.020195       1.374459       1.368897
  O MSE=1.238484 | C MSE=1.382129 | E MSE=1.079457 | A MSE=1.804299 | N MSE=1.367925
block6                   0.019509  

Probe Seed 42 Val 11/50: 100%|██████████| 191/191 [00:20<00:00,  9.18it/s]
[2026-09-25 01:49:40,107]::169859163::INFO::Probe early stopping | seed=42 | epoch=11 | best mean MSE=1.363757


\nSeed 42 | Epoch 011 | Mean Val MSE=1.384577 | Best Mean Val MSE=1.363757 | Patience=10/10
Representation          Train MSE        Val MSE       Best MSE
----------------------------------------------------------------
block1                   0.046821       1.467374       1.442900
  O MSE=1.359418 | C MSE=1.290955 | E MSE=1.313201 | A MSE=1.811216 | N MSE=1.562083
block2                   0.032026       1.432511       1.400695
  O MSE=1.334962 | C MSE=1.429727 | E MSE=1.074521 | A MSE=1.866328 | N MSE=1.457016
block3                   0.024476       1.419056       1.391495
  O MSE=1.300615 | C MSE=1.395686 | E MSE=1.091442 | A MSE=1.879991 | N MSE=1.427547
block4                   0.022122       1.400333       1.371801
  O MSE=1.264311 | C MSE=1.380750 | E MSE=1.109690 | A MSE=1.823972 | N MSE=1.422940
block5                   0.019688       1.391539       1.368897
  O MSE=1.253357 | C MSE=1.388048 | E MSE=1.108799 | A MSE=1.822875 | N MSE=1.384618
block6                   0.019075 

/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
